# Replication Sample

Sample ~1000 prompts equally across every **domain × condition** combination for the replication/consistency test. Each sampled prompt will be sent to every model 5 times by `reproduction.py`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('/data/jane/convert/math_gender/conversion_test')
PREPROC_DIR = PROJECT_ROOT / 'full_results' / 'preprocessed'

CONDITION_MAP = {
    'regular':   {'suffix': '',            'condition_label': 'in_domain_with_guide'},
    'no_guide':  {'suffix': '_no_guide',   'condition_label': 'in_domain_no_guide'},
    'math_only': {'suffix': '_math_only',  'condition_label': 'math_only'},
}

DOMAINS = [
    'bits_bytes', 'cooking', 'currency', 'density', 'energy',
    'moles_to_particles', 'speed', 'temperature', 'timezone', 'volume',
]

TARGET_TOTAL = 1000
rng = np.random.RandomState(42)

In [ ]:
# Discover all domain × condition combos and count unique prompts in each
combos = []
for domain in DOMAINS:
    for cond_key, cond_info in CONDITION_MAP.items():
        fname = f"{domain}{cond_info['suffix']}.tsv"
        fpath = PREPROC_DIR / fname
        if not fpath.exists():
            continue
        df = pd.read_csv(fpath, sep='\t')
        combos.append({
            'domain': domain,
            'condition_key': cond_key,
            'condition_label': cond_info['condition_label'],
            'file': str(fpath),
            'n_prompts': len(df),
        })

combo_df = pd.DataFrame(combos)
n_combos = len(combo_df)
per_combo = TARGET_TOTAL // n_combos

print(f"Found {n_combos} domain × condition combos")
print(f"Sampling {per_combo} prompts per combo → {per_combo * n_combos} total")
print()
print(combo_df[['domain','condition_label','n_prompts']].to_string(index=False))

Found 30 domain × condition combos
Sampling 33 prompts per combo → 990 total

            domain      condition_label  n_prompts
        bits_bytes in_domain_with_guide       1200
        bits_bytes   in_domain_no_guide       1200
        bits_bytes            math_only       1200
           cooking in_domain_with_guide      37200
           cooking   in_domain_no_guide      37200
           cooking            math_only       1200
          currency in_domain_with_guide      22400
          currency   in_domain_no_guide      22400
          currency            math_only      22400
           density in_domain_with_guide      37200
           density   in_domain_no_guide      37200
           density            math_only       1200
            energy in_domain_with_guide      62000
            energy   in_domain_no_guide      62000
            energy            math_only       2000
moles_to_particles in_domain_with_guide      12400
moles_to_particles   in_domain_no_guide      12400
mole

In [ ]:
# Sample from each combo
samples = []
for _, row in combo_df.iterrows():
    df = pd.read_csv(row['file'], sep='\t')
    n = min(per_combo, len(df))
    chosen = df.sample(n=n, random_state=rng)
    chosen = chosen[['domain', 'distractor', 'prompt', 'number', 'answer']].copy()
    chosen['condition'] = row['condition_label']
    chosen['condition_key'] = row['condition_key']
    samples.append(chosen)

sample_df = pd.concat(samples, ignore_index=True)
print(f"Total sampled prompts: {len(sample_df)}")
print()
print("Per domain × condition:")
print(sample_df.groupby(['domain','condition']).size().unstack(fill_value=0).to_string())

Total sampled prompts: 990

Per domain × condition:
condition           in_domain_no_guide  in_domain_with_guide  math_only
domain                                                                 
bits_bytes                          33                    33         33
cooking                             33                    33         33
currency                            33                    33         33
density                             33                    33         33
energy                              33                    33         33
moles_to_particles                  33                    33         33
speed                               33                    33         33
temperature                         33                    33         33
timezone                            33                    33         33
volume                              33                    33         33


In [ ]:
# Save the sample
OUT_PATH = PROJECT_ROOT / '4_replication_test' / 'replication_sample.tsv'
sample_df.to_csv(OUT_PATH, sep='\t', index=False)
print(f"Saved to {OUT_PATH}")
print(f"Columns: {sample_df.columns.tolist()}")
print(f"\nSample rows:")
sample_df.head(10)

Saved to /data/jane/convert/math_gender/conversion_test/4_replication_test/replication_sample.tsv
Columns: ['domain', 'distractor', 'prompt', 'number', 'answer', 'condition', 'condition_key']

Sample rows:


,domain,distractor,prompt,number,answer,condition,condition_key
0,bits_bytes,NaN,Convert 273.15 bits to megabytes.\n\nConversio...,273.15,0.000034,in_domain_with_guide,regular
1,bits_bytes,NaN,Convert 280 megabytes to gigabytes.\n\nConvers...,280.0,0.28,in_domain_with_guide,regular
2,bits_bytes,NaN,Convert 1297.195 bits to bytes.\n\nConversion ...,1297.195,162.149375,in_domain_with_guide,regular
3,bits_bytes,NaN,Convert 70 bits to kilobytes.\n\nConversion gu...,70.0,0.00875,in_domain_with_guide,regular
4,bits_bytes,NaN,Convert 200 bits to bytes.\n\nConversion guide...,200.0,25.0,in_domain_with_guide,regular
5,bits_bytes,NaN,Convert 763.9 bits to megabytes.\n\nConversion...,763.9,0.000095,in_domain_with_guide,regular
6,bits_bytes,NaN,Convert 8472.3 bytes to gigabytes.\n\nConversi...,8472.3,0.000008,in_domain_with_guide,regular
7,bits_bytes,NaN,Convert 638.29 megabytes to gigabytes.\n\nConv...,638.29,0.63829,in_domain_with_guide,regular
8,bits_bytes,NaN,Convert 12 bits to kilobytes.\n\nConversion gu...,12.0,0.0015,in_domain_with_guide,regular
9,bits_bytes,NaN,Convert 160 megabytes to gigabytes.\n\nConvers...,160.0,0.16,in_domain_with_guide,regular
